# Can we predict late delivery?
Target: `late_delivery`. Prediction is for future consignments, so the split is temporal. Each consignment ID appears once, so no entity grouping is needed (checked below).

Feature decisions (from columns.csv): `consignment` is an identifier, so it is dropped. `claim_paid_eur` is recorded after delivery, when a claim is settled, and is a consequence of lateness, so it is dropped as leakage. Features kept: carrier, distance_km, weight_kg, parcels, and dispatch hour / weekday (known at dispatch). Calendar month/trend is not used as a feature.

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
df=pd.read_csv('data.csv',parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
print(df.consignment.is_unique, len(df))
df['hour']=df.dispatched_on.dt.hour; df['dow']=df.dispatched_on.dt.dayofweek
FEATS=['carrier','distance_km','weight_kg','parcels','hour','dow']

## 1. Hold out the test set (latest 20% of rows by dispatch time)

In [ ]:
cut=int(len(df)*0.8)
train,test=df.iloc[:cut].copy(),df.iloc[cut:].copy()
print(train.dispatched_on.max(), test.dispatched_on.min())
print('train late rate',train.late_delivery.mean(),'test rows',len(test))
X,y=train[FEATS],train.late_delivery
monthly=train.groupby(train.dispatched_on.dt.to_period('M')).late_delivery.mean(); print(monthly.round(3).to_string())

## 2. Candidate models (named before any scoring)
1. `baseline`: predicts the training prior (trivial)
2. `logreg`: logistic regression (scaled numerics, one-hot carrier)
3. `hgb`: gradient boosting, small grid over `max_depth` in {2,3} and `learning_rate` in {0.05,0.1}

Selection metric: mean PR-AUC across expanding-window time-series CV folds in the training rows only. Decision threshold: chosen on pooled out-of-fold predictions to maximise F1. Preprocessing lives inside the pipelines, so it is fitted within each fold.

In [ ]:
def pre(): return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),['carrier']),('n',StandardScaler(),['distance_km','weight_kg','parcels','hour','dow'])])
cands={'baseline':lambda:DummyClassifier(strategy='prior'),
 'logreg':lambda:make_pipeline(pre(),LogisticRegression(max_iter=1000))}
for d in (2,3):
    for lr in (0.05,0.1):
        cands[f'hgb_d{d}_lr{lr}']=(lambda d=d,lr=lr:make_pipeline(pre(),HistGradientBoostingClassifier(max_depth=d,learning_rate=lr,max_iter=150,random_state=0)))
cv=TimeSeriesSplit(n_splits=5)
res={};oof={}
for name,mk in cands.items():
    p=np.full(len(train),np.nan); sc=[]
    for tr,va in cv.split(X):
        m=mk().fit(X.iloc[tr],y.iloc[tr]); p[va]=m.predict_proba(X.iloc[va])[:,1]
        sc.append((average_precision_score(y.iloc[va],p[va]),roc_auc_score(y.iloc[va],p[va])))
    oof[name]=p; res[name]=np.mean(sc,axis=0)
cvres=pd.DataFrame(res,index=['cv_pr_auc','cv_roc_auc']).T.sort_values('cv_pr_auc',ascending=False); cvres

In [ ]:
best=cvres.drop('baseline').index[0]
mask=~np.isnan(oof[best]); ths=np.linspace(0.05,0.9,86)
f1s=[f1_score(y[mask],oof[best][mask]>t) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
print(best,'threshold',thr,'oof F1',max(f1s))

## 3. Choice (from cross-validation only, recorded before the test set is scored)
The chosen model is the top non-baseline CV PR-AUC above, with the threshold from the cell above.

In [ ]:
print('CHOSEN:',best,'threshold',thr)
print(cvres.loc[['baseline',best]])

## 4. Refit on all training rows; score the test set once

In [ ]:
m=cands[best]().fit(X,y); pt=m.predict_proba(test[FEATS])[:,1]; yt=test.late_delivery
b=cands['baseline']().fit(X,y).predict_proba(test[FEATS])[:,1]
out={'test_rows':len(test),'test_late_rate':yt.mean(),'PR-AUC':average_precision_score(yt,pt),'ROC-AUC':roc_auc_score(yt,pt),
 'F1@thr':f1_score(yt,pt>thr),'Brier':brier_score_loss(yt,pt),'baseline_PR-AUC(=prevalence)':average_precision_score(yt,b),'baseline_Brier':brier_score_loss(yt,b)}
pd.Series(out)

## Interpretation
See the headline test numbers above. Caveat: the late rate swings strongly over time (roughly 30% early, falling to ~1% by 2026), so the test period sits in a different regime from training. The features carry limited information about that drift, so calibration and the threshold may transfer poorly even if ranking holds.